[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marclamberts/mm-setpieces-1/blob/main/notebooks/fetch_corners_colab.ipynb)

# Fetch Corners Data → Save to GitHub

Pulls corner-kick events from the StatsBomb API, links each delivery to any shot  
in the same possession, derives `SP_outcome`, and pushes one parquet per competition  
directly to `Data/Corners/` in the GitHub repo.

**Steps:**
1. Install deps  
2. Enter your GitHub token  
3. Run all cells (`Runtime → Run all`)

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q statsbombpy pyarrow fastparquet

In [ ]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
import getpass

# StatsBomb credentials
ACCOUNTS = [
    {"user": "m.pulley@az.nl",         "passwd": "SwHrVcks"},
    {"user": "JACK71299@HOTMAIL.CO.UK", "passwd": "J7rB7aP2"},
]

# GitHub — enter your Personal Access Token when prompted
# (needs repo write access: Settings → Developer settings → Personal access tokens)
GITHUB_TOKEN = getpass.getpass("GitHub Personal Access Token: ")
GITHUB_REPO  = "marclamberts/mm-setpieces-1"   # owner/repo
GITHUB_BRANCH = "main"

# Seasons to fetch
TARGET_SEASON_IDS = [316, 318]
SEASON_LABELS = {
    316: "2026",
    318: "2025-2026",
}

MAX_WORKERS   = 6
SKIP_EXISTING = True   # skip if parquet already exists in GitHub

print("Config ready.")

In [ ]:
# ── 3. Imports ────────────────────────────────────────────────────────────────
import base64
import io
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from statsbombpy import sb

print("Imports OK.")

In [ ]:
# ── 4. GitHub helpers ─────────────────────────────────────────────────────────

GH_API = "https://api.github.com"
GH_HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
}


def gh_file_exists(path: str) -> tuple[bool, str | None]:
    """Return (exists, sha) for a file in the repo."""
    url = f"{GH_API}/repos/{GITHUB_REPO}/contents/{path}?ref={GITHUB_BRANCH}"
    r = requests.get(url, headers=GH_HEADERS, timeout=30)
    if r.status_code == 200:
        return True, r.json().get("sha")
    return False, None


def gh_push_bytes(content_bytes: bytes, repo_path: str, commit_msg: str) -> bool:
    """Create or update a file in the repo with raw bytes content."""
    exists, sha = gh_file_exists(repo_path)
    url = f"{GH_API}/repos/{GITHUB_REPO}/contents/{repo_path}"
    payload = {
        "message": commit_msg,
        "content": base64.b64encode(content_bytes).decode(),
        "branch":  GITHUB_BRANCH,
    }
    if exists:
        payload["sha"] = sha
    r = requests.put(url, headers=GH_HEADERS, json=payload, timeout=120)
    if r.status_code in (200, 201):
        return True
    print(f"  GitHub push failed ({r.status_code}): {r.text[:200]}")
    return False


def gh_get_raw(repo_path: str) -> bytes | None:
    """Download raw bytes of a file from the repo."""
    url = f"{GH_API}/repos/{GITHUB_REPO}/contents/{repo_path}?ref={GITHUB_BRANCH}"
    r = requests.get(url, headers=GH_HEADERS, timeout=60)
    if r.status_code == 200:
        return base64.b64decode(r.json()["content"])
    return None


def safe_filename(name: str) -> str:
    return re.sub(r'[<>:"/\\|?*\t]', '', name).strip()


# Verify token works
resp = requests.get(f"{GH_API}/repos/{GITHUB_REPO}", headers=GH_HEADERS)
print(f"GitHub connection: {'OK — ' + resp.json().get('full_name', '') if resp.ok else 'FAILED — ' + str(resp.status_code)}")

In [ ]:
# ── 5. StatsBomb helpers ──────────────────────────────────────────────────────

def log(msg: str) -> None:
    print(msg, flush=True)


def try_accounts(func, *args, **kwargs):
    for idx, creds in enumerate(ACCOUNTS, 1):
        try:
            result = func(*args, creds=creds, **kwargs)
            if isinstance(result, pd.DataFrame) and not result.empty:
                return result
            if isinstance(result, (list, dict)) and result:
                return result
            if result is not None:
                return result
        except Exception as e:
            log(f"  Account {idx} failed ({func.__name__}): {e}")
    return None


def ts_to_seconds(ts) -> float:
    if pd.isna(ts):
        return np.nan
    parts = str(ts).split(":")
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])


print("StatsBomb helpers ready.")

In [ ]:
# ── 6. SP_outcome logic ───────────────────────────────────────────────────────
#
#  pass.outcome.name  |  shot?  |  timing    → SP_outcome
#  Out                |  —      |  —         → Wayward
#  Unknown            |  —      |  —         → 50/50
#  Incomplete         |  —      |  —         → First contact lost
#  NaN (complete)     |  Yes    |  ≤ 3 s     → First contact - shot within 3 seconds
#  NaN (complete)     |  Yes    |  > 3 s     → No first contact - shot
#  NaN (complete)     |  No     |  end_x>102 → Short in area
#  NaN (complete)     |  No     |  otherwise → No first contact - no shot

PENALTY_BOX_X         = 102.0
FIRST_CONTACT_SECS    = 3.0


def derive_sp_outcome(row) -> str:
    outcome  = row.get("pass.outcome.name")
    has_shot = pd.notna(row.get("shot.outcome.name"))

    if outcome == "Out":        return "Wayward"
    if outcome == "Unknown":    return "50/50"
    if outcome == "Incomplete": return "First contact lost"

    if has_shot:
        dt = row.get("_dt", np.nan)
        if pd.notna(dt) and dt <= FIRST_CONTACT_SECS:
            return "First contact - shot within 3 seconds"
        return "No first contact - shot"

    end_x = row.get("pass_end_location_x", np.nan)
    if pd.notna(end_x) and end_x > PENALTY_BOX_X:
        return "Short in area"
    return "No first contact - no shot"


print("SP_outcome logic ready.")

In [ ]:
# ── 7. Per-match extraction ───────────────────────────────────────────────────

OUTPUT_COLS = [
    "match_id", "Match", "possession",
    "pass_timestamp", "pass_team_name", "Taker", "pass_position",
    "pass.height.name", "pass.body_part.name", "pass.outcome.name", "pass.technique.name",
    "pass_location_x", "pass_location_y", "pass_end_location_x", "pass_end_location_y",
    "shot_timestamp", "shot_team_name", "Shooter", "shot_position",
    "shot.body_part.name", "shot.outcome.name", "shot.statsbomb_xg",
    "shot_location_x", "shot_location_y", "shot_location_z",
    "Defensive_setup", "Minute", "Second", "SP_outcome",
]


def get_nested(series, *keys):
    result = series
    for key in keys:
        result = result.apply(lambda x: x.get(key) if isinstance(x, dict) else np.nan)
    return result


def extract_corners_from_match(match_id: int, match_label: str) -> pd.DataFrame | None:
    raw = try_accounts(sb.events, match_id=int(match_id))
    if raw is None or (isinstance(raw, pd.DataFrame) and raw.empty):
        return None
    if not isinstance(raw, pd.DataFrame):
        try:
            raw = pd.DataFrame(raw)
        except Exception:
            return None

    # Corner passes
    passes = raw[raw["type"].apply(lambda x: x.get("name") if isinstance(x, dict) else x) == "Pass"].copy()
    if "pass" not in passes.columns:
        return None
    corners = passes[get_nested(passes["pass"], "type", "name") == "Corner"].copy()
    if corners.empty:
        return None

    # Flatten pass fields
    corners["pass_timestamp"]       = corners["timestamp"]
    corners["pass_team_name"]       = corners["team"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
    corners["Taker"]                = corners["player"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
    corners["pass_position"]        = corners["position"].apply(lambda x: x.get("name") if isinstance(x, dict) else np.nan)
    corners["pass.height.name"]     = get_nested(corners["pass"], "height", "name")
    corners["pass.body_part.name"]  = get_nested(corners["pass"], "body_part", "name")
    corners["pass.outcome.name"]    = get_nested(corners["pass"], "outcome", "name")
    corners["pass.technique.name"]  = get_nested(corners["pass"], "technique", "name")
    corners["pass_location_x"]      = corners["location"].apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else np.nan)
    corners["pass_location_y"]      = corners["location"].apply(lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else np.nan)
    end_loc                         = corners["pass"].apply(lambda x: x.get("end_location") if isinstance(x, dict) else None)
    corners["pass_end_location_x"]  = end_loc.apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else np.nan)
    corners["pass_end_location_y"]  = end_loc.apply(lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else np.nan)
    corners["Minute"]               = corners["minute"]
    corners["Second"]               = corners["second"]
    corners["match_id"]             = match_id
    corners["Match"]                = match_label

    # Link first shot in the same possession
    shots = raw[raw["type"].apply(lambda x: x.get("name") if isinstance(x, dict) else x) == "Shot"].copy()
    if not shots.empty and "shot" in shots.columns:
        shots["shot_team_name"]      = shots["team"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
        shots["Shooter"]             = shots["player"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
        shots["shot_position"]       = shots["position"].apply(lambda x: x.get("name") if isinstance(x, dict) else np.nan)
        shots["shot.body_part.name"] = get_nested(shots["shot"], "body_part", "name")
        shots["shot.outcome.name"]   = get_nested(shots["shot"], "outcome", "name")
        shots["shot.statsbomb_xg"]   = shots["shot"].apply(lambda x: x.get("statsbomb_xg") if isinstance(x, dict) else np.nan)
        shots["shot_location_x"]     = shots["location"].apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 1 else np.nan)
        shots["shot_location_y"]     = shots["location"].apply(lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else np.nan)
        shots["shot_location_z"]     = shots["location"].apply(lambda x: x[2] if isinstance(x, list) and len(x) >= 3 else np.nan)
        shots["shot_timestamp"]      = shots["timestamp"]
        shot_cols = [c for c in ["possession", "shot_team_name", "Shooter", "shot_position",
                                  "shot.body_part.name", "shot.outcome.name", "shot.statsbomb_xg",
                                  "shot_location_x", "shot_location_y", "shot_location_z",
                                  "shot_timestamp"] if c in shots.columns]
        corners = corners.merge(
            shots[shot_cols].drop_duplicates(subset=["possession"], keep="first"),
            on="possession", how="left",
        )
    else:
        for col in ["shot_team_name", "Shooter", "shot_position", "shot.body_part.name",
                    "shot.outcome.name", "shot.statsbomb_xg", "shot_location_x",
                    "shot_location_y", "shot_location_z", "shot_timestamp"]:
            corners[col] = np.nan

    corners["Defensive_setup"] = ""

    # SP_outcome
    corners["_dt"] = (
        corners["shot_timestamp"].apply(ts_to_seconds) -
        corners["pass_timestamp"].apply(ts_to_seconds)
    )
    corners["SP_outcome"] = corners.apply(derive_sp_outcome, axis=1)

    for col in OUTPUT_COLS:
        if col not in corners.columns:
            corners[col] = np.nan

    return corners[OUTPUT_COLS].reset_index(drop=True)


print("extract_corners_from_match ready.")

In [ ]:
# ── 8. Main loop — fetch, build parquet, push to GitHub ───────────────────────

# Load competition list from GitHub
log("Loading competitions.csv from GitHub ...")
comps_bytes = gh_get_raw("Data/competitions.csv")
if comps_bytes:
    all_comps = pd.read_csv(io.BytesIO(comps_bytes))
    log(f"Loaded {len(all_comps)} competitions.")
else:
    log("Not found in GitHub — fetching from StatsBomb ...")
    all_comps = try_accounts(sb.competitions)
    if all_comps is None:
        raise SystemExit("Could not load competitions.")

targets = (
    all_comps[all_comps["season_id"].isin(TARGET_SEASON_IDS)]
    .drop_duplicates(subset=["competition_id", "season_id"])
    .copy()
)
log(f"Competitions to process: {len(targets)}\n")

ok_count = skipped = failed = 0

for _, row in targets.iterrows():
    comp_id    = int(row["competition_id"])
    season_id  = int(row["season_id"])
    comp_name  = str(row.get("competition_name", f"Competition_{comp_id}"))
    season_lbl = SEASON_LABELS.get(season_id, str(season_id))
    fname      = f"{safe_filename(comp_name)} - Corners {season_lbl}.parquet"
    gh_path    = f"Data/Corners/{fname}"

    log(f"→ [{season_id}] {comp_name}")

    if SKIP_EXISTING:
        exists, _ = gh_file_exists(gh_path)
        if exists:
            log(f"  [skip] already in GitHub")
            skipped += 1
            continue

    # Fetch match list
    matches = try_accounts(sb.matches, competition_id=comp_id, season_id=season_id)
    if matches is None or (isinstance(matches, pd.DataFrame) and matches.empty):
        log("  ✗ no matches found")
        failed += 1
        continue

    matches["match_label"] = (
        matches["home_team"].apply(lambda x: x.get("home_team_name") if isinstance(x, dict) else str(x))
        + " - "
        + matches["away_team"].apply(lambda x: x.get("away_team_name") if isinstance(x, dict) else str(x))
    )
    match_ids = matches["match_id"].dropna().astype(int).tolist()
    label_map = dict(zip(matches["match_id"].astype(int), matches["match_label"]))
    log(f"  Fetching {len(match_ids)} matches ...")

    # Parallel fetch
    frames = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(extract_corners_from_match, mid, label_map.get(mid, str(mid))): mid
                   for mid in match_ids}
        for fut in as_completed(futures):
            try:
                df = fut.result()
                if df is not None and not df.empty:
                    frames.append(df)
            except Exception as exc:
                log(f"    match {futures[fut]} error: {exc}")

    if not frames:
        log("  ✗ no corner data")
        failed += 1
        continue

    combined = pd.concat(frames, ignore_index=True)

    # Serialise to parquet bytes in memory
    buf = io.BytesIO()
    combined.to_parquet(buf, engine="pyarrow", index=False)
    buf.seek(0)
    parquet_bytes = buf.read()

    # Push to GitHub
    commit_msg = f"Add {fname} ({len(combined):,} corners) via Colab fetch"
    success = gh_push_bytes(parquet_bytes, gh_path, commit_msg)
    if success:
        log(f"  ✓ pushed {len(combined):,} corners → Data/Corners/{fname}")
        ok_count += 1
    else:
        log(f"  ✗ GitHub push failed")
        failed += 1

    time.sleep(1)   # be polite to the API

log(f"\n{'='*60}")
log(f"Done — pushed: {ok_count}  |  skipped: {skipped}  |  failed/empty: {failed}")

In [ ]:
# ── 9. Verify — list files now in GitHub ─────────────────────────────────────

url = f"{GH_API}/repos/{GITHUB_REPO}/contents/Data/Corners?ref={GITHUB_BRANCH}"
r = requests.get(url, headers=GH_HEADERS)

if r.ok:
    files = r.json()
    print(f"{len(files)} parquet files in Data/Corners/:")
    for f in sorted(files, key=lambda x: x['name']):
        size_kb = f.get('size', 0) // 1024
        print(f"  {f['name']}  ({size_kb} KB)")
else:
    print(f"Could not list directory: {r.status_code}")